# 01 — Data Fetching & Exploration

This notebook downloads price data for the asset universe, computes returns, and explores their statistical properties.

**Contents:**
1. Download prices from Yahoo Finance
2. Compute log returns
3. Descriptive statistics & annualized metrics
4. Correlation matrix
5. Distribution analysis (normality test)

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats

from src.data_fetcher.fetcher import download_prices, compute_returns, describe_returns
from src.utils.plotting import plot_correlation_heatmap
from config import TICKERS, START_DATE, PRICE_FREQUENCY, RETURN_TYPE

pd.set_option('display.float_format', '{:.4f}'.format)
print('Libraries loaded ✓')

## 1. Download Price Data

In [ ]:
prices = download_prices(
    tickers=TICKERS,
    start=START_DATE,
    interval=PRICE_FREQUENCY,
    cache_dir='../data/',
)
print(f'Shape: {prices.shape}')
prices.tail()

In [ ]:
# Plot normalized prices (rebased to 100)
fig, ax = plt.subplots(figsize=(12, 5))
(prices / prices.iloc[0] * 100).plot(ax=ax, lw=1.5)
ax.set_title('Normalized Prices (Base=100)', fontsize=13)
ax.set_ylabel('Index')
plt.tight_layout()
plt.savefig('../results/01_prices.png', dpi=150)
plt.show()

## 2. Compute Returns

In [ ]:
returns = compute_returns(prices, method=RETURN_TYPE, winsorize_sigma=3.0)
print(f'Returns shape: {returns.shape}')
returns.tail()

## 3. Descriptive Statistics

In [ ]:
stats_df = describe_returns(returns)
print('\n=== Annualized Statistics ===')
display(stats_df)

## 4. Correlation Matrix

In [ ]:
fig = plot_correlation_heatmap(returns, save_path='../results/01_correlation.png')
plt.show()
print('Note: Low/negative correlations between equity and bonds are the foundation of diversification.')

## 5. Normality Test (Jarque-Bera)

Markowitz assumes normal returns. Let's check how far reality deviates.

In [ ]:
print('Jarque-Bera Normality Test (p-value < 0.05 → reject normality)\n')
for col in returns.columns:
    jb_stat, p_val = stats.jarque_bera(returns[col].dropna())
    flag = '❌ Non-normal' if p_val < 0.05 else '✓ Normal'
    print(f'  {col:6s}: stat={jb_stat:7.2f}, p={p_val:.4f}  {flag}')

In [ ]:
# Return distribution plots
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for i, (col, ax) in enumerate(zip(returns.columns, axes.flat)):
    ax.hist(returns[col], bins=30, color='#2196F3', edgecolor='white', alpha=0.8, density=True)
    x = np.linspace(returns[col].min(), returns[col].max(), 100)
    ax.plot(x, stats.norm.pdf(x, returns[col].mean(), returns[col].std()), 'r--', lw=1.5, label='Normal')
    ax.set_title(col, fontsize=10)
    ax.legend(fontsize=7)
plt.suptitle('Return Distributions vs Normal', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/01_distributions.png', dpi=150)
plt.show()

print('\n→ Fat tails justify the use of CVaR over VaR/variance as risk measure.')

In [ ]:
returns.to_csv('../data/returns.csv')
print('Returns saved to ../data/returns.csv ✓')